In [ ]:
# Colab / fresh environment: install the module (builds the C++ core, takes a few minutes the first time)
try:
    import sensor_fusion
except ImportError:
    %pip install -q git+https://github.com/Rudip1/learn-sensor-fusion

# 4 · Point clouds: representation, voxel grids, k-d trees, normals

**Recap** ([theory](../../1_theory/04_point_clouds.md)). A LiDAR returns the first hit along each ray (4.1);
a scan is an $n \times 3$ array in the sensor frame. A voxel grid keys points by $\lfloor(\mathbf p-\mathbf o)/\ell\rfloor$ (4.2)
and keeps one centroid per voxel (4.3). A k-d tree answers nearest-neighbour queries in $O(\log n)$ (Algorithm 4.2).
The normal is the eigenvector of the smallest eigenvalue of the neighbourhood covariance (4.4)–(4.5); the surface
variation (4.6) flags edges and degenerate neighbourhoods.

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

import sensor_fusion as sf
from sensor_fusion import cloud, sim
from sensor_fusion.plotting import SERIES, use_style

use_style()
rng = np.random.default_rng(4)

T = np.eye(4)
T[:3, 3] = [0.0, 0.0, 1.8]                          # sensor 1.8 m above the ground
model = sim.LidarModel.spinning(32, -25.0, 10.0, 1024)  # 32 beams from -25 to +10 degrees
model.range_noise = 0.02
scan = sim.simulate_lidar(sim.demo_scene(), T, model, seed=1)
P, labels, true_n = scan["points"], scan["labels"], scan["normals"]
print(f"{len(P)} points, {np.mean(labels == -1):.0%} on the ground, ranges {np.linalg.norm(P, axis=1).min():.1f}-{np.linalg.norm(P, axis=1).max():.1f} m")

## 1. A recorded scan from a PCD file

In [ ]:
d = cloud.read_pcd(str(sf.data.path("parking_lot/run1_scans.pcd")))
print({k: v.shape for k, v in d.items()})
first = d["scan"] == 0
fig, ax = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
ax[0].scatter(d["x"][first], d["y"][first], s=3, c=d["intensity"][first]); ax[0].plot(0, 0, "k^")
ax[0].set_aspect("equal"); ax[0].set_title("run 1, scan 0 (planar, coloured by intensity)")
ax[1].hist(np.hypot(d["x"], d["y"]), bins=60); ax[1].set_xlabel("range [m]"); ax[1].set_title(f"{int(d['scan'].max()) + 1} scans")
plt.show()

### ✏️ Exercise 1 — voxel grid
Implement `my_voxel(points, leaf)` returning the centroids with eqs. (4.2)–(4.3) (origin 0), in order of first
occurrence, and compare with `cloud.voxel_downsample`.

In [ ]:
def my_voxel(points, leaf):
    # ✏️ your code here
    raise NotImplementedError


In [ ]:
for leaf in (0.2, 0.5, 1.0):
    mine = my_voxel(P, leaf)
    ref = cloud.voxel_downsample(P, leaf)[0]
    assert mine.shape == ref.shape and np.allclose(mine, ref)
    print(f"leaf {leaf} m: {len(ref)} points ({len(ref) / len(P):.0%})")
# floor, not truncation: points either side of 0 must not share a voxel
assert len(my_voxel(np.array([[-0.3, 0, 0], [0.3, 0, 0]]), 1.0)) == 2
print("✅ my_voxel matches the C++ voxel grid")

## 2. Nearest neighbours

### ✏️ Exercise 2 — brute force, the reference
Implement `brute_knn(points, q, k)` returning (indices, distances), sorted by distance (the convention of
`tree.knn` in Python; the C++ API returns squared distances).

In [ ]:
def brute_knn(points, q, k):
    # ✏️ your code here
    raise NotImplementedError


In [ ]:
tree = cloud.KdTree(P)
queries = P[rng.choice(len(P), 300)] + rng.normal(0, 0.1, (300, 3))
for q in queries:
    i1, d1 = brute_knn(P, q, 8)
    i2, d2 = tree.knn(q, 8)
    assert np.allclose(d1, d2)
print(f"✅ the k-d tree (depth {tree.depth}) agrees with brute force on 300 queries")

t0 = time.perf_counter(); [tree.knn(q, 10) for q in queries]; t_tree = time.perf_counter() - t0
t0 = time.perf_counter(); [brute_knn(P, q, 10) for q in queries]; t_brute = time.perf_counter() - t0
print(f"300 queries: k-d tree {1000 * t_tree:.1f} ms, NumPy brute force {1000 * t_brute:.1f} ms")

## 3. Normals

### ✏️ Exercise 3 — PCA normal
Implement `pca_normal(neigh)` returning (normal, surface variation) with eqs. (4.4)–(4.6).

In [ ]:
def pca_normal(neigh):
    # ✏️ your code here
    raise NotImplementedError


In [ ]:
R = np.linalg.qr(rng.normal(size=(3, 3)))[0]
pts = np.c_[rng.uniform(-1, 1, (50, 2)), np.zeros(50)] @ R.T
n, var = pca_normal(pts)
assert abs(abs(n @ R[:, 2]) - 1) < 1e-12 and var < 1e-12
ref_n, _, ref_var = cloud.fit_plane(pts)
assert abs(abs(n @ ref_n) - 1) < 1e-12 and abs(var - ref_var) < 1e-12
print("✅ pca_normal: exact on a plane, same as cloud.fit_plane")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.6), constrained_layout=True)
for i, k in enumerate((5, 10, 20, 40)):
    n_est = cloud.estimate_normals(P, k, np.zeros(3))[0]
    ang = np.degrees(np.arccos(np.clip(np.abs(np.sum(n_est * true_n, axis=1)), 0, 1)))
    ax.plot(np.sort(ang), np.linspace(0, 1, len(ang)), color=SERIES[i], label=f"k = {k}: median {np.median(ang):.1f}°")
ax.set_xscale("log"); ax.set_xlim(0.05, 90); ax.set_xlabel("error [deg]"); ax.legend(); plt.show()

## 🔨 Break it — a neighbourhood on one ring

On the ground the nearest neighbours of a point with small $k$ all lie on its own ring: a line. Show it with the
eigenvalues.

In [ ]:
g = np.flatnonzero(labels == -1)[2000]
for k in (5, 40):
    idx, _ = tree.knn(P[g], k)
    w = np.linalg.eigvalsh(np.cov(P[idx].T, bias=True))
    n, var = pca_normal(P[idx])
    print(f"k = {k:2d}: eigenvalues {np.array2string(w, precision=6)}, normal {np.round(n, 2)} (truth {np.round(true_n[g], 2)})")

## 🔨 Break it — truncation instead of floor

`astype(int)` rounds towards zero, so the voxel around the origin is twice as wide.

In [ ]:
x = np.linspace(-0.99, 0.99, 9)[:, None] * np.ones((1, 3))
print("floor keys:", np.floor(x[:, 0]).astype(int))
print("trunc keys:", x[:, 0].astype(int))

## 🔨 Break it — downsample first, then normals

With a 0.5 m voxel grid and k = 20, neighbourhoods are metres wide. Compare the errors on the ground and on the
objects (cars, buildings, posts).

In [ ]:
cent, vox = cloud.voxel_downsample(P, 0.5)[:2]
n_sparse = cloud.estimate_normals(cent, 20, np.zeros(3))[0][vox]   # normal of each point's voxel
n_dense = cloud.estimate_normals(P, 20, np.zeros(3))[0]
err = lambda n: np.degrees(np.arccos(np.clip(np.abs(np.sum(n * true_n, axis=1)), 0, 1)))
obj = labels >= 0
for name, e in (("dense", err(n_dense)), ("0.5 m grid", err(n_sparse))):
    print(f"{name:10s}: median {np.median(e):4.1f} deg;  on objects {np.mean(e[obj] > 10):.0%} off by more than 10 deg")

Better on average (the grid removes the ring anisotropy of the ground), much worse near edges.

## What to remember

- A scan is dense near the sensor and anisotropic (fine along rings, coarse across them).
- Voxel grids give uniform density; key with `floor`, keep centroids.
- Build one k-d tree per cloud; every neighbour query is then $O(\log n)$.
- PCA normals need neighbourhoods wide enough for the noise and spanning two rings, but within one surface;
  the surface variation tells which normals to trust.